In [1]:
# ================================
# IMPORTS
# ================================
import json
import os
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer


# ================================
# PATHS
# ================================
faiss_index_path = "data/vector_store/faiss_index.bin"
metadata_path = "data/vector_store/metadata.json"


# ================================
# LOAD INDEX
# ================================
index = faiss.read_index(faiss_index_path)

with open(metadata_path, "r", encoding="utf-8") as f:
    metadata = json.load(f)

print(f"Loaded {len(metadata)} metadata entries")


# ================================
# LOAD SAME EMBEDDING MODEL 🔥
# ================================
print("\n🔹 Loading embedding model...")
model = SentenceTransformer("BAAI/bge-small-en-v1.5")


# ================================
# QUERY PREFIX + EXPANSION 🔥
# ================================
def process_query(query):
    # expand query
    expanded = query + " fertilizer soil nutrients crop farming agriculture"

    # add prefix (important for BGE)
    return "Represent this sentence for searching relevant passages: " + expanded


# ================================
# RETRIEVAL FUNCTION (IMPROVED)
# ================================
def retrieve(query, top_k=8, domain_filter=None):

    query = process_query(query)

    query_embedding = model.encode([query], convert_to_numpy=True)
    faiss.normalize_L2(query_embedding)

    # search more results for filtering
    distances, indices = index.search(query_embedding, top_k * 3)

    results = []
    seen_texts = set()

    for i, idx in enumerate(indices[0]):

        if idx >= len(metadata):
            continue

        item = metadata[idx]
        score = float(distances[0][i])

        # 🔥 Domain boosting
        if domain_filter and item["domain"] == domain_filter:
            score *= 1.2

        # 🔥 Remove duplicates (MMR-style)
        if item["text"] in seen_texts:
            continue

        seen_texts.add(item["text"])

        results.append({
            "rank": len(results) + 1,
            "score": score,
            "text": item["text"],
            "source": item["source"],
            "domain": item["domain"]
        })

        if len(results) >= top_k:
            break

    return results


# ================================
# TEST RETRIEVAL
# ================================
query = "how to improve soil fertility for wheat"

print("\n🔹 Query:\n", query)

results = retrieve(query, top_k=8, domain_filter="soil")

print("\n🔹 Retrieved Chunks:\n")

for res in results:
    print(f"Rank {res['rank']} | Score: {res['score']:.4f}")
    print(f"Domain: {res['domain']}")
    print(f"Source: {res['source']}")
    print(res["text"][:200])
    print("\n---------------------------\n")

e:\Udemy ML course\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded 148 metadata entries

🔹 Loading embedding model...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3031.45it/s]
BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



🔹 Query:
 how to improve soil fertility for wheat

🔹 Retrieved Chunks:

Rank 1 | Score: 0.8073
Domain: fertilizer
Source: https://aksharfarmtech.com/blog/proven-ways-to-boost-crop-yield-naturally/
your cart is currently empty. proven ways to boost crop yield naturally introductionevery farmer dreams of a thriving, high-yielding farm. but with soil depletion, unpredictable weather, and increasin

---------------------------

Rank 2 | Score: 0.7981
Domain: fertilizer
Source: https://www.fertilizer.org/wp-content/uploads/2023/01/2016_ifa_reetz.pdf
animal feed and forage, industrial products, energy and for an aesthetically pleasing environment. soil fertility integrates the basic principles of soil biology, soil chemistry, and soil physics to d

---------------------------

Rank 3 | Score: 0.7739
Domain: fertilizer
Source: https://www.fertilizer.org/wp-content/uploads/2023/01/2016_ifa_reetz.pdf
if new lands are available these are often less productive. the need will probably be met by a